# Slogan Generator and Industry Classifier using LSTM Neural Networks

## Project Overview

This project explores Natural Language Processing (NLP) by developing two Long Short-Term Memory (LSTM) neural networks using TensorFlow.

The first model generates industry-specific marketing slogans by learning language patterns from existing slogans. The second model classifies a slogan into its corresponding industry, demonstrating how recurrent neural networks can be applied to text classification tasks.

## Objectives

- Preprocess text using spaCy
- Generate numerical sequences from text data
- Train an LSTM model to generate marketing slogans
- Train an LSTM classifier to predict slogan industries
- Evaluate model performance and generate sample predictions

## Technologies Used

- Python
- TensorFlow / Keras
- spaCy
- Pandas
- NumPy
- Scikit-learn

## Import Libraries

In [23]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam

import spacy

from sklearn.model_selection import train_test_split

## Load the Dataset

The dataset contains company slogans, their associated companies, and the industry each company belongs to. These data form the basis of both tasks in this project: generating new marketing slogans and classifying slogans by industry.

After loading the dataset, only the relevant columns are retained, missing values are removed, and the index is reset to ensure a clean dataset for model training.

In [24]:
# Load dataset
df = pd.read_csv("slogan-valid.csv")

# Extract relevant columns and handle missing values
df = df[["output", "industry", "company"]].copy()
df = df.dropna().reset_index(drop=True)

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"Number of industries: {df['industry'].nunique()}")
print(f"Number of companies: {df['company'].nunique()}")

df.head()

Dataset shape: (5346, 3)
Number of industries: 142
Number of companies: 5346


,output,industry,company
0,Taking Care of Small Business Technology,computer hardware,eftpos warehouse
1,Build World-Class Recreation Programs,"health, wellness and fitness",welbi
2,Most Powerful Lead Generation Software for Mar...,internet,optinmonster
3,Hire quality freelancers for your job,internet,twine.fm
4,"Financial Advisers Norwich, Norfolk",financial services,mcb financial services ltd


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. The generated input sequences have different lengths.

In [25]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

# Define text preprocessing function
def preprocess_text(text):
    # Convert all text to lowercase
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,output,industry,company,processed_slogan
0,Taking Care of Small Business Technology,computer hardware,eftpos warehouse,taking care of small business technology
1,Build World-Class Recreation Programs,"health, wellness and fitness",welbi,build world class recreation programs
2,Most Powerful Lead Generation Software for Mar...,internet,optinmonster,most powerful lead generation software for mar...
3,Hire quality freelancers for your job,internet,twine.fm,hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,mcb financial services ltd,financial advisers norwich norfolk


## Create Industry-Specific Input

The goal of the slogan generation model is to produce slogans that are relevant to a particular industry. If the model is trained using only the processed slogans, it has no information about the industry each slogan belongs to.

To provide this context, the industry label is prepended to each processed slogan, creating a new `modified_slogan` column. During training, the model learns the relationship between an industry and the language commonly used in its marketing slogans, enabling it to generate more industry-specific outputs.

For example:

- **Industry:** `computer hardware`
- **Processed slogan:** `taking care of small business technology`
- **Modified slogan:** `computer hardware taking care of small business technology`

In [26]:
# Append the industry name to each processed slogan
df["modified_slogan"] = df["industry"] + " " + df["processed_slogan"]
df[["industry", "processed_slogan", "modified_slogan"]].head()

,industry,processed_slogan,modified_slogan
0,computer hardware,taking care of small business technology,computer hardware taking care of small busines...
1,"health, wellness and fitness",build world class recreation programs,"health, wellness and fitness build world class..."
2,internet,most powerful lead generation software for mar...,internet most powerful lead generation softwar...
3,internet,hire quality freelancers for your job,internet hire quality freelancers for your job
4,financial services,financial advisers norwich norfolk,financial services financial advisers norwich ...


## Tokenization and Sequence Generation

Neural networks cannot process raw text directly, so each slogan must first be converted into a numerical representation.

In this step, a tokenizer is fitted to the `modified_slogan` column to build a vocabulary of all unique words. Each word is assigned a unique integer, allowing every slogan to be represented as a sequence of numbers.

These numerical sequences are then used to create training examples for the LSTM model. Each training sequence consists of an input sequence and the next word the model is expected to predict, enabling the network to learn the patterns and structure of marketing slogans.

In [27]:
# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index
tokenizer.word_index

# Creating input sequences
input_sequences = []

for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0]

    # Building list of progressively longer input sequences
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

print(f"Vocabulary size: {len(tokenizer.word_index)}")

Vocabulary size: 6045


## Pad Input Sequences

The generated training sequences vary in length because each slogan contains a different number of words. However, LSTM networks require all input sequences to have a consistent length.

To prepare the data for training, each sequence is padded with leading zeros until it matches the length of the longest sequence. This standardization ensures that all input sequences have the same dimensions while preserving the original word order.

In [28]:
# Find the length of the longest sequence
max_seq_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length: {max_seq_len}")

Maximum sequence length: 15


In [29]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Prepare Training Data for the Slogan Generator

After padding, each sequence is separated into predictor variables (`X`) and a target label (`y`). The predictor variables contain the input words, while the target label represents the next word the model should learn to predict.

The target labels are then one-hot encoded so they can be used to train the LSTM model for multi-class word prediction.

In [30]:
# Split input sequences into inputs (X) and targets (y)
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

print(f"Generator input shape: {X_gen.shape}")
print(f"Generator target shape: {y_gen.shape}")

Generator input shape: (34736, 14)
Generator target shape: (34736,)


In [31]:
# One-hot encode the target word indices
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)
print(f"One-hot encoded target shape: {y_gen.shape}")

One-hot encoded target shape: (34736, 6046)


## Build the LSTM Slogan Generator

The slogan generator is built using a Long Short-Term Memory (LSTM) neural network. LSTMs are well suited to language generation because they can learn relationships between words across a sequence and retain information from earlier words when predicting the next one.

The model consists of:

- **Embedding layer:** Converts each word into a dense vector representation that captures semantic relationships between words.
- **Stacked LSTM layers**: Learn complex sequential patterns and contextual relationships between words.
- **Dense output layer:** Produces a probability distribution over the vocabulary, allowing the model to predict the most likely next word.

The model is trained to predict one word at a time, enabling it to generate complete slogans by repeatedly predicting the next word in a sequence.

In [32]:
# Build LSTM model
gen_model = tf.keras.models.Sequential([
    # Embedding layer
    Embedding(total_words, 100),

    # First LSTM layer
    LSTM(150, return_sequences=True),

    # Second LSTM layer
    LSTM(100),

    # Dense output layer
    Dense(total_words, activation="softmax")
])

## Compile the Model

The model is compiled using the Adam optimizer and categorical cross-entropy loss. Since the model predicts one word from the entire vocabulary, this is treated as a multi-class classification problem. Accuracy is included as an evaluation metric to monitor the model's performance during training.

In [33]:
# Compile the generator model
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
    )

gen_model.build(input_shape=(None, max_seq_len - 1))
gen_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 14, 100)             │         604,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 14, 150)             │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 100)                 │         100,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 6046)                │         610,646 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,466,246 (5.59 MB)

 Trainable params: 1,466,246 (5.59 MB)

 Non-trainable params: 0 (0.00 B)

The model architecture contains approximately **1.47 million trainable parameters**. The embedding layer learns dense word representations, the stacked LSTM layers capture sequential language patterns, and the dense output layer predicts the next word from the full vocabulary.

## Train the Slogan Generator

The model is trained on the prepared slogan sequences using categorical cross-entropy loss. During training, it learns to predict the next word in a slogan based on the preceding words, gradually capturing common language patterns and industry-specific terminology.

In [34]:
gen_model.fit(X_gen, y_gen, epochs=50, verbose=1)

Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 80s 65ms/step - accuracy: 0.0717 - loss: 7.0506
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 69s 64ms/step - accuracy: 0.1014 - loss: 6.3822
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 68s 63ms/step - accuracy: 0.1401 - loss: 6.0427
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 71s 65ms/step - accuracy: 0.1702 - loss: 5.7746
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 70s 65ms/step - accuracy: 0.1975 - loss: 5.5248
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 91s 73ms/step - accuracy: 0.2153 - loss: 5.2971
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 81s 74ms/step - accuracy: 0.2282 - loss: 5.0897
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 79s 72ms/step - accuracy: 0.2400 - loss: 4.8898
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 74s 65ms/step - accuracy: 0.2506 - loss: 4.6993
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 70s 64ms/step - accuracy: 0.2582 - loss: 4.5226
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 82s 64ms/step - accuracy: 0.2668 - loss: 4.3479
Epoch 12

After 50 epochs, the model learned meaningful language patterns from the training slogans. Although the model is capable of generating industry-related text, the quality of the generated slogans depends on the size and diversity of the training dataset, and longer sequences may become less coherent.

## Generate Industry-Specific Slogans

After training the LSTM model, new slogans can be generated by predicting one word at a time.

The generation process begins with a seed phrase, which in this project is typically an industry name. The model predicts the most likely next word, appends it to the input sequence, and then repeats the process until a complete slogan has been generated or the maximum number of words has been reached.

This iterative approach allows the model to produce slogans that reflect the language patterns it learned during training.

In [49]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        # Convert the current text into a padded sequence of integer tokens
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")

        # Predict the probability of each word in the vocabulary
        predictions = gen_model.predict(token_list, verbose=0)

        # Select the most probable next word
        predicted_index = np.argmax(predictions[0])

        output_word = None

        # Searching for the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # If no valid word is found, algorithm stops
        if output_word is None:
            break # out of main loop

        # Append the predicted word to the generated slogan
        seed_text += " " + output_word

    return seed_text

## Prepare Training Data for the Industry Classifier

The second part of this project focuses on predicting a company's industry based on its slogan.

The classifier uses the `processed_slogan` column as the input features and the `industry` column as the target labels. Before training the model, each unique industry is assigned a numerical identifier, allowing the classifier to work with integer-encoded class labels instead of text.

In [50]:
# Get the sorted list of unique industries
industries = sorted(df["industry"].unique())
print(f"Total unique industries: {len(industries)}")
print(industries[:10])

Total unique industries: 142
['accounting', 'airlines/aviation', 'alternative medicine', 'animation', 'apparel & fashion', 'architecture & planning', 'arts and crafts', 'automotive', 'aviation & aerospace', 'banking']


### Encode Industry Labels

Each industry is mapped to a unique integer value. These encoded labels serve as the target classes that the classifier will learn to predict.

In [51]:
# Map each unique industry name to a unique integer index
industry_to_index = {industry: index for index, industry in enumerate(industries)}

# Display a sample of the industry-to-index mappings
dict(list(industry_to_index.items())[:5])

{'accounting': 0,
 'airlines/aviation': 1,
 'alternative medicine': 2,
 'animation': 3,
 'apparel & fashion': 4}

In [52]:
# Add a new column to df with the integer index for each industry
df["industry_index"] = df["industry"].map(industry_to_index)

# Preview the updated DataFrame
df.head()

,output,industry,company,processed_slogan,modified_slogan,industry_index
0,Taking Care of Small Business Technology,computer hardware,eftpos warehouse,taking care of small business technology,computer hardware taking care of small busines...,21
1,Build World-Class Recreation Programs,"health, wellness and fitness",welbi,build world class recreation programs,"health, wellness and fitness build world class...",52
2,Most Powerful Lead Generation Software for Mar...,internet,optinmonster,most powerful lead generation software for mar...,internet most powerful lead generation softwar...,65
3,Hire quality freelancers for your job,internet,twine.fm,hire quality freelancers for your job,internet hire quality freelancers for your job,65
4,"Financial Advisers Norwich, Norfolk",financial services,mcb financial services ltd,financial advisers norwich norfolk,financial services financial advisers norwich ...,41


## Split the Dataset

Before training the classifier, the dataset is divided into training and testing sets. A stratified split is used to preserve the distribution of industries in both subsets, ensuring that the evaluation data remains representative of the original dataset.

Industries represented by only a single sample are removed prior to splitting, as stratified sampling requires at least two observations per class.

In [53]:
# Identify industries represented by only a single slogan
industry_counts = df["industry_index"].value_counts()
single_sample_industries = industry_counts[industry_counts < 2].index

# Remove industries that cannot be used for stratified sampling
df_filtered = df[~df["industry_index"].isin(single_sample_industries)]

# Split dataframe into training and test sets
df_train, df_test = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered["industry_index"]
)

print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape: {df_test.shape}")

Training set shape: (4272, 6)
Testing set shape: (1068, 6)


## Prepare the Input Sequences

The classifier uses complete processed slogans as input rather than the progressively growing sequences used by the slogan generator. Since the objective is to predict the industry from an entire slogan, each slogan is converted into a sequence of integer tokens using the tokenizer created earlier.

The same tokenizer is applied to both the training and testing datasets to ensure that words are represented consistently throughout the classification process.

In [54]:
# Convert processed slogans to sequences of numerical indices
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

## Pad the Input Sequences

The tokenized slogans vary in length because different slogans contain different numbers of words. To provide the neural network with inputs of consistent dimensions, each sequence is padded with leading zeros until it matches the maximum sequence length used during training.

In [55]:
# Pad sequences so they all have the same length
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre")
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre")

## Encode the Target Labels

The industry labels are converted into one-hot encoded vectors so they can be used as target outputs during model training. Each vector contains a single positive value corresponding to the correct industry class, while all other positions remain zero.

In [56]:
# One-hot encode industry indices for both training and testing sets
y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"],
    num_classes=len(industries)
)

y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"],
    num_classes=len(industries)
)

print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

y_train shape: (4272, 142)
y_test shape:  (1068, 142)


## Build the Industry Classifier

The industry classifier is implemented using a Long Short-Term Memory (LSTM) neural network. The model learns to identify patterns in complete company slogans and use these patterns to predict the corresponding industry.

The model consists of:

- **Embedding layer:** Converts each word into a dense vector representation that captures semantic relationships.
- **LSTM layer:** Learns sequential language patterns within the slogans.
- **Dense output layer:** Produces a probability distribution over all industry classes, allowing the model to predict the most likely industry.

In [57]:
# Build the LSTM slogan classifier model
class_model = tf.keras.models.Sequential([
    # Embedding layer
    Embedding(total_words, 100),

    # First LSTM layer
    LSTM(150, return_sequences=True),

    # Second LSTM layer
    LSTM(100),

    # Dense output layer
    Dense(len(industries), activation="softmax")
])

## Compile the Industry Classifier

The model is compiled using the Adam optimizer and categorical cross-entropy loss. Since the industry labels have been one-hot encoded, categorical cross-entropy is an appropriate loss function. Accuracy is included as the evaluation metric to monitor the classifier's performance during training.

In [58]:
# Compile the classifier model
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

class_model.build((None, max_seq_len))
class_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)              │ (None, 15, 100)             │         604,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_8 (LSTM)                        │ (None, 15, 150)             │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_9 (LSTM)                        │ (None, 100)                 │         100,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 142)                 │          14,342 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 869,942 (3.32 MB)

 Trainable params: 869,942 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

The summary above confirms that the classifier consists of an embedding layer, two LSTM layers, and a dense output layer. Together, these layers contain 869,942 trainable parameters that are optimized during training to learn the relationship between slogan text and industry labels.

The final output layer contains 142 neurons, corresponding to the 142 industry categories in the dataset.

## Train the Industry Classifier

The classifier is trained using the prepared slogan sequences and their corresponding industry labels. During training, the model learns to identify language patterns that distinguish one industry from another by adjusting its parameters over multiple epochs.

In [59]:
# Train the slogan classifier for 50 epochs
class_model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 25s 83ms/step - accuracy: 0.0866 - loss: 4.3876 - val_accuracy: 0.0843 - val_loss: 4.2823
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 69ms/step - accuracy: 0.0847 - loss: 4.2885 - val_accuracy: 0.0843 - val_loss: 4.2715
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 10s 70ms/step - accuracy: 0.0847 - loss: 4.2536 - val_accuracy: 0.0787 - val_loss: 4.1729
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.1206 - loss: 3.9716 - val_accuracy: 0.1245 - val_loss: 4.0061
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.1983 - loss: 3.5782 - val_accuracy: 0.1704 - val_loss: 3.9149
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.2795 - loss: 3.1688 - val_accuracy: 0.1891 - val_loss: 3.9086
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.3727 - loss: 2.7715 - val_accuracy: 0.1845 - val_loss: 3.9625
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.4452 - loss: 2.4273 - va

The industry classification model was trained on the processed slogan dataset for 50 epochs. During training, the model learned to associate patterns in slogan text with their corresponding industry categories.

The next step is to evaluate the model on the held-out test dataset to assess how well it generalises to unseen slogans.

## Evaluate the Industry Classifier

The trained classifier is evaluated on the testing dataset to measure how well it generalises to unseen slogans. The reported accuracy provides an indication of the model's predictive performance on new data.

In [60]:
# Evaluate the classifier on the test set
loss, accuracy = class_model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# The model achieved 18.16% accuracy across 142 industry classes.
# This is relatively low, which is expected given that business slogans are
# intentionally short and vague.

# The test loss of 7.72 is high, which is common when dealing with a large
# number of classes and limited training data per class.

Test Loss: 7.4687
Test Accuracy: 0.2004


The classifier achieved a **test accuracy of 20.04%** on the held-out test dataset. While the model learned meaningful relationships between slogan text and industry categories, classifying slogans remains a challenging task due to the large number of industry classes (142) and the overlap in language used across many industries.

The relatively high test loss and moderate accuracy suggest that the model has difficulty distinguishing between industries with similar vocabulary. Nevertheless, the classifier performs substantially better than random guessing and demonstrates the feasibility of applying LSTM networks to multi-class text classification.

## Predict the Industry of a Slogan

The function below preprocesses a slogan, converts it into a padded sequence of numerical tokens, and passes it to the trained classifier. The model returns the probability of each industry, and the industry with the highest probability is selected as the final prediction.

In [61]:
def classify_slogan(slogan):
    # Use the preprocess_text to clean the input slogan (we defined this function in the preprocessing section)
    slogan = preprocess_text(slogan)

    # Converting the slogan into a sequence of integer tokens
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence to match the input length expected by the model
    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len, padding="pre")

    # Predict the probability of each industry
    prediction = class_model.predict(padded_sequence, verbose=0)

    # Select the industry with the highest predicted probability
    predicted_index = np.argmax(prediction[0])

    # Return the predicted industry
    return industries[predicted_index]

## Combine the Two Models

The final stage of the project demonstrates how the slogan generator and industry classifier can be used together.

First, the slogan generator creates a new slogan from a given industry prompt. The generated slogan is then passed to the industry classifier, which predicts the industry based solely on the generated text.

This provides an end-to-end demonstration of both models working together within a single natural language processing pipeline.

In [64]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

Generated Slogan: internet web design development agency surry hills tx home of containers and networks up to ireland and madison expertise perth simple
Predicted Industry: marketing and advertising


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


## Discussion

The generated slogan begins with words that are relevant to the input industry, such as *web*, *design*, *development*, and *networks*. However, the sequence gradually becomes less coherent as unrelated words are introduced. This behaviour is typical of LSTM-based language models trained on relatively small datasets.

The classifier predicted **Marketing and Advertising** instead of the original **Internet** industry. Although the prediction is incorrect, it is understandable because the generated slogan contains several terms commonly associated with digital marketing and web agencies.

The classifier achieved a test accuracy of approximately **20%** across **142 industry categories**. While this demonstrates that the model learned meaningful language patterns, the large number of classes and the similarity between many industries make accurate classification challenging. Additionally, classification performance is affected by the quality of the generated slogans, which are not always fully coherent.

## Limitations and Future Improvements

This project demonstrates the complete workflow for text generation and classification using LSTM networks. However, several improvements could increase performance:

- Train on a larger and more diverse slogan dataset.
- Use pretrained word embeddings (e.g., GloVe or Word2Vec).
- Replace the LSTM with a transformer-based language model for improved text generation.
- Tune hyperparameters such as the number of LSTM units, learning rate, and batch size.
- Train for longer or apply early stopping based on validation performance.